# 1. Initialisation et Recensement des Données


In [10]:
import os
import pandas as pd

# Chemin vers les donnees brutes
dossier_brut = "../data/raw"

print("Fichiers detectes :")
for racine, dossiers, fichiers in os.walk(dossier_brut):
    for fichier in fichiers:
        if fichier.endswith('.csv'):
            print(os.path.join(racine, fichier))


Fichiers detectes :
../data/raw/PhiUSIIL_Phishing_URL_Dataset.csv
../data/raw/Phishing_Legitimate_full.csv
../data/raw/phishings_emails/Ling.csv
../data/raw/phishings_emails/Nigerian_Fraud.csv
../data/raw/phishings_emails/Enron.csv
../data/raw/phishings_emails/Nazario.csv
../data/raw/phishings_emails/phishing_email.csv
../data/raw/phishings_emails/CEAS_08.csv
../data/raw/phishings_emails/SpamAssasin.csv


# 2. Analyse du Dataset des URLs Brutes
Chargement du nouveau fichier contenant les chaines de caracteres des URLs pour verifier leur structure et l'equilibre des classes.


In [11]:
# Chargement du nouveau dataset d'URLs
chemin_urls_brutes = os.path.join(dossier_brut, "PhiUSIIL_Phishing_URL_Dataset.csv")
df_urls = pd.read_csv(chemin_urls_brutes)

print(f"Dimensions du dataset URLs : {df_urls.shape}")
print("Colonnes disponibles :", df_urls.columns.tolist()[:5])

# Identification de la colonne contenant l'URL brute textuelle
# Dans PhiUSIIL, la colonne de l'adresse brute s'appelle generalement 'URL'
nom_colonne_url = 'URL' if 'URL' in df_urls.columns else df_urls.columns[1]
print(f"Colonne retenue pour le texte de l'URL : {nom_colonne_url}")

# Normalisation de la cible : 1 = phishing, 0 = legitime
# La pipeline de production mappe le label source 0 vers phishing (1).
df_urls['cible'] = df_urls['label'].apply(lambda x: 1 if x == 0 else 0)

print("\nDistribution des classes :")
print(df_urls['cible'].value_counts())
print("\nPourcentage des classes :")
print(df_urls['cible'].value_counts(normalize=True) * 100)


Dimensions du dataset URLs : (235795, 56)
Colonnes disponibles : ['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength']
Colonne retenue pour le texte de l'URL : URL

Distribution des classes :
cible
1    134850
0    100945
Name: count, dtype: int64

Pourcentage des classes :
cible
1    57.189508
0    42.810492
Name: proportion, dtype: float64


# 3. Analyse du Dataset des Emails


In [12]:
# Chargement du fichier consolide des emails
chemin_emails = os.path.join(dossier_brut, "phishings_emails/phishing_email.csv")
df_emails = pd.read_csv(chemin_emails)

print(f"Dimensions du dataset Emails : {df_emails.shape}")
print("Colonnes disponibles :", df_emails.columns.tolist())

# Fixation manuelle des bonnes colonnes detectees
colonne_texte = 'text_combined'
colonne_cible = 'label'

print("\nDistribution des classes :")
print(df_emails[colonne_cible].value_counts())
print("\nPourcentage des classes :")
print(df_emails[colonne_cible].value_counts(normalize=True) * 100)

# Verification des valeurs manquantes sur la bonne colonne
print(f"\nLignes avec du texte manquant : {df_emails[colonne_texte].isnull().sum()}")


Dimensions du dataset Emails : (82486, 2)
Colonnes disponibles : ['text_combined', 'label']

Distribution des classes :
label
1    42891
0    39595
Name: count, dtype: int64

Pourcentage des classes :
label
1    51.997915
0    48.002085
Name: proportion, dtype: float64

Lignes avec du texte manquant : 0


# 4. Verification de la consolidation des emails


In [13]:
import glob

# Liste de tous les fichiers csv individuels
tous_fichiers = glob.glob(os.path.join(dossier_brut, "phishings_emails/*.csv"))

total_lignes_individuelles = 0
for chemin in tous_fichiers:
    nom_fichier = os.path.basename(chemin)
    if nom_fichier != "phishing_email.csv":
        df_temporaire = pd.read_csv(chemin)
        print(f"Fichier : {nom_fichier} | Lignes : {df_temporaire.shape[0]}")
        total_lignes_individuelles += df_temporaire.shape[0]

print(f"\nTotal cumule des fichiers individuels : {total_lignes_individuelles}")
print(f"Total dans le fichier unique phishing_email.csv : {df_emails.shape[0]}")


Fichier : Ling.csv | Lignes : 2859
Fichier : Nigerian_Fraud.csv | Lignes : 3332
Fichier : Enron.csv | Lignes : 29767
Fichier : Nazario.csv | Lignes : 1565
Fichier : CEAS_08.csv | Lignes : 39154
Fichier : SpamAssasin.csv | Lignes : 5809

Total cumule des fichiers individuels : 82486
Total dans le fichier unique phishing_email.csv : 82486


# 5. Analyse Comparative des Longueurs de Texte
Analyse de la longueur des emails et des URLs selon leur classe pour identifier si la taille est un indicateur discriminant.


In [14]:
# Calcul des longueurs sur la vraie colonne textuelle
df_urls['longueur_url'] = df_urls[nom_colonne_url].astype(str).apply(len)

print("Longueur moyenne des URLs par classe :")
print(df_urls.groupby('cible')['longueur_url'].mean())


Longueur moyenne des URLs par classe :
cible
0    46.238774
1    27.228610
Name: longueur_url, dtype: float64


# 6. Analyse des Valeurs Manquantes et Doublons
Identification et comptage précis des lignes corrompues ou dupliquées à nettoyer avant la phase de feature engineering.


In [15]:
# Analyse des doublons et des valeurs manquantes
total_doublons_url = df_urls[nom_colonne_url].duplicated().sum()
total_doublons_email = df_emails['text_combined'].duplicated().sum()

total_manquants_url = df_urls[nom_colonne_url].isnull().sum()
total_manquants_email = df_emails['text_combined'].isnull().sum()

print("Doublons URLs :", total_doublons_url)
print("Doublons Emails :", total_doublons_email)
print("Valeurs manquantes URLs :", total_manquants_url)
print("Valeurs manquantes Emails :", total_manquants_email)


Doublons URLs : 425
Doublons Emails : 408
Valeurs manquantes URLs : 0
Valeurs manquantes Emails : 0


# 7. Analyse Initiale des Indices Cyber sur les URLs
Vérification de la présence de caractères suspects (. ou -) dans les URLs selon leur classe (0 = Sain, 1 = Phishing).


In [16]:
# Comptage des points sur la vraie colonne textuelle
df_urls['nb_points'] = df_urls[nom_colonne_url].astype(str).apply(lambda x: x.count('.'))

print("Nombre moyen de points par classe :")
print(df_urls.groupby('cible')['nb_points'].mean())


Nombre moyen de points par classe :
cible
0    2.384695
1    2.161661
Name: nb_points, dtype: float64
